In [ ]:
!pip install -q qiskit qiskit-machine-learning kagglehub torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.1/263.1 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 106.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 5.7 MB/s eta 0:00:00


In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from kagglehub import dataset_download
from sklearn.metrics import classification_report, confusion_matrix

from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector
from qiskit import QuantumCircuit

In [ ]:
IMAGE_SIZE = 224
BATCH_SIZE = 16
FEATURE_DIM = 8          # ⚠️ keep ≤ 8 for feasibility
EPOCHS = 1
LR = 1e-4

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
dataset_root = dataset_download("thilak02/breast-cancer-detection-using-thermography")

dataset_path = os.path.join(dataset_root, "BCD_Dataset")
#valid_path = os.path.join(dataset_root, "Valid", "Valid")
#test_path  = os.path.join(dataset_root, "test", "test")

print(dataset_path)
print(os.listdir(dataset_path))

100%|██████████| 1.39M/1.39M [00:01<00:00, 1.34MB/s]

Extracting files...
/root/.cache/kagglehub/datasets/thilak02/breast-cancer-detection-using-thermography/versions/1/BCD_Dataset
['Sick', 'normal', 'Unknown_class']


In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import Dataset, random_split, DataLoader
from collections import Counter
import torch

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

# Base dataset
base_dataset = datasets.ImageFolder(dataset_path, transform=transform)
print("Original classes:", base_dataset.classes)

# -----------------------------
# Filter Unknown_class
# -----------------------------
class FilteredDataset(Dataset):
    def __init__(self, base_dataset, ignore_class="Unknown"):
        self.samples = []
        self.class_names = ["normal", "Sick"]

        orig_classes = base_dataset.classes
        print("Original dataset classes:", orig_classes)

        ignore_idx = orig_classes.index(ignore_class)

        for img, label in base_dataset:
            if label == ignore_idx:
                continue

            class_name = orig_classes[label].lower()

            if class_name in ["healthy", "normal"]:
                new_label = 0
            elif class_name in ["cancer", "sick", "malignant"]:
                new_label = 1
            else:
                continue   # safety net

            self.samples.append((img, new_label))

        if len(self.samples) == 0:
            raise RuntimeError("Filtered dataset is empty!")

        print(f"Filtered dataset size: {len(self.samples)}")
        print("Binary classes:", self.class_names)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

filtered_dataset = FilteredDataset(base_dataset, ignore_class="Unknown_class")

# -----------------------------
# Train / Val split (ONLY filtered)
# -----------------------------
train_size = int(0.8 * len(filtered_dataset))
val_size   = len(filtered_dataset) - train_size

generator = torch.Generator().manual_seed(42)
train_ds, val_ds = random_split(
    filtered_dataset,
    [train_size, val_size],
    generator=generator
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

# -----------------------------
# Class counting (correct)
# -----------------------------
def count_classes(subset, class_names):
    counter = Counter()
    for _, label in subset:
        counter[class_names[label]] += 1
    return dict(counter)

class_names = filtered_dataset.class_names

train_counts = count_classes(train_ds, class_names)
val_counts   = count_classes(val_ds, class_names)

print("\nTraining dataset counts:")
for cls in class_names:
    print(f"  {cls}: {train_counts.get(cls, 0)}")

print("\nValidation dataset counts:")
for cls in class_names:
    print(f"  {cls}: {val_counts.get(cls, 0)}")

Original classes: ['Sick', 'Unknown_class', 'normal']
Original dataset classes: ['Sick', 'Unknown_class', 'normal']
Filtered dataset size: 262
Binary classes: ['normal', 'Sick']

Training dataset counts:
  normal: 127
  Sick: 82

Validation dataset counts:
  normal: 35
  Sick: 18


In [ ]:
feature_map = ZZFeatureMap(feature_dimension=FEATURE_DIM, reps=1)
ansatz = RealAmplitudes(num_qubits=FEATURE_DIM, reps=1)

qc = QuantumCircuit(FEATURE_DIM)
qc.compose(feature_map, inplace=True)
qc.compose(ansatz, inplace=True)

qnn = EstimatorQNN(
    circuit=qc,
    input_params=feature_map.parameters,
    weight_params=ansatz.parameters
)

/tmp/ipython-input-86803684.py:1: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(feature_dimension=FEATURE_DIM, reps=1)
/tmp/ipython-input-86803684.py:2: DeprecationWarning: The class ``qiskit.circuit.library.n_local.real_amplitudes.RealAmplitudes`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.real_amplitudes instead.
  ansatz = RealAmplitudes(num_qubits=FEATURE_DIM, reps=1)


In [ ]:
qc.decompose().draw()

┌───┐┌───────────┐                                             »
q_0: ┤ H ├┤ P(2*x[0]) ├──■──────────────────────────────────■────■──»
     ├───┤├───────────┤┌─┴─┐┌────────────────────────────┐┌─┴─┐  │  »
q_1: ┤ H ├┤ P(2*x[1]) ├┤ X ├┤ P(2*(π - x[0])*(π - x[1])) ├┤ X ├──┼──»
     ├───┤├───────────┤└───┘└────────────────────────────┘└───┘┌─┴─┐»
q_2: ┤ H ├┤ P(2*x[2]) ├────────────────────────────────────────┤ X ├»
     ├───┤├───────────┤                                        └───┘»
q_3: ┤ H ├┤ P(2*x[3]) ├─────────────────────────────────────────────»
     ├───┤├───────────┤                                             »
q_4: ┤ H ├┤ P(2*x[4]) ├─────────────────────────────────────────────»
     ├───┤├───────────┤                                             »
q_5: ┤ H ├┤ P(2*x[5]) ├─────────────────────────────────────────────»
     ├───┤├───────────┤                                             »
q_6: ┤ H ├┤ P(2*x[6]) ├─────────────────────────────────────────────»
     ├───┤├───────────┤                                             »
q_7: ┤ H ├┤ P(2*x[7]) ├─────────────────────────────────────────────»
     └───┘└───────────┘                                             »
«                                                  »
«q_0: ────────────────────────────────■─────────■──»
«                                     │         │  »
«q_1: ────────────────────────────────┼────■────┼──»
«     ┌────────────────────────────┐┌─┴─┐┌─┴─┐  │  »
«q_2: ┤ P(2*(π - x[0])*(π - x[2])) ├┤ X ├┤ X ├──┼──»
«     └────────────────────────────┘└───┘└───┘┌─┴─┐»
«q_3: ────────────────────────────────────────┤ X ├»
«                                             └───┘»
«q_4: ─────────────────────────────────────────────»
«                                                  »
«q_5: ─────────────────────────────────────────────»
«                                                  »
«q_6: ─────────────────────────────────────────────»
«                                                  »
«q_7: ─────────────────────────────────────────────»
«                                                  »
«                                                       »
«q_0: ─────────────────────────────────────■─────────■──»
«                                          │         │  »
«q_1: ────────────────────────────────■────┼────■────┼──»
«     ┌────────────────────────────┐┌─┴─┐  │    │    │  »
«q_2: ┤ P(2*(π - x[1])*(π - x[2])) ├┤ X ├──┼────┼────┼──»
«     ├────────────────────────────┤└───┘┌─┴─┐┌─┴─┐  │  »
«q_3: ┤ P(2*(π - x[0])*(π - x[3])) ├─────┤ X ├┤ X ├──┼──»
«     └────────────────────────────┘     └───┘└───┘┌─┴─┐»
«q_4: ─────────────────────────────────────────────┤ X ├»
«                                                  └───┘»
«q_5: ──────────────────────────────────────────────────»
«                                                       »
«q_6: ──────────────────────────────────────────────────»
«                                                       »
«q_7: ──────────────────────────────────────────────────»
«                                                       »
«                                                            »
«q_0: ─────────────────────────────────────■──────────────■──»
«                                          │              │  »
«q_1: ────────────────────────────────■────┼─────────■────┼──»
«                                     │    │         │    │  »
«q_2: ────────────────────────────────┼────┼────■────┼────┼──»
«     ┌────────────────────────────┐┌─┴─┐  │  ┌─┴─┐  │    │  »
«q_3: ┤ P(2*(π - x[1])*(π - x[3])) ├┤ X ├──┼──┤ X ├──┼────┼──»
«     ├────────────────────────────┤└───┘┌─┴─┐└───┘┌─┴─┐  │  »
«q_4: ┤ P(2*(π - x[0])*(π - x[4])) ├─────┤ X ├─────┤ X ├──┼──»
«     └────────────────────────────┘     └───┘     └───┘┌─┴─┐»
«q_5: ──────────────────────────────────────────────────┤ X ├»
«                                                       └───┘»
«q_6: ───────────────────────────────────────────────────────»
«                                             

In [ ]:
class HybridEndToEnd(nn.Module):
    def __init__(self, qnn, feature_dim):
        super().__init__()

        # ---------------------------
        # Classical CNN (VGG19)
        # ---------------------------
        self.cnn = models.vgg19(pretrained=True)

        # Remove VGG classifier
        self.cnn.classifier = nn.Identity()

        # Feature bottleneck → QNN
        self.reduce = nn.Sequential(
            nn.Linear(25088, 128),
            nn.ReLU(),
            nn.Linear(128, feature_dim),
            nn.Tanh()
        )

        # ---------------------------
        # Quantum Layer
        # ---------------------------
        self.qnn = TorchConnector(qnn)

        # Final classifier
        self.fc = nn.Linear(1, 1)

    def forward(self, x):
        x = self.cnn(x)
        x = self.reduce(x)
        x = self.qnn(x)
        x = self.fc(x)
        return x          # LOGITS


In [ ]:
model = HybridEndToEnd(qnn, FEATURE_DIM).to(device)

# Freeze VGG weights for stability
for param in model.cnn.parameters():
    param.requires_grad = False

# Leave bottleneck + QNN trainable

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


100%|██████████| 548M/548M [00:02<00:00, 203MB/s]


In [ ]:
#criterion = nn.BCELoss()  #this is binary cross-entropy loss
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)

In [ ]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    TP = FP = TN = FN = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.float().view(-1, 1).to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = (outputs >= 0.5).float()

        TP += ((preds == 1) & (labels == 1)).sum().item()
        TN += ((preds == 0) & (labels == 0)).sum().item()
        FP += ((preds == 1) & (labels == 0)).sum().item()
        FN += ((preds == 0) & (labels == 1)).sum().item()

    accuracy = 100 * (TP + TN) / (TP + TN + FP + FN)
    precision = TP / (TP + FP + 1e-8)
    recall = TP / (TP + FN + 1e-8)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Loss: {total_loss / len(train_loader):.4f} | "
        f"Acc: {accuracy:.2f}% | "
        f"Prec: {precision:.3f} | "
        f"Recall: {recall:.3f}"
    )
    torch.save(model,'breast-thermography_vgg_qnn_checkpoint.pt')

Epoch 1/1 | Loss: 0.7367 | Acc: 60.77% | Prec: 0.000 | Recall: 0.000


In [ ]:
torch.save(model,'breast-thermography_vgg_qnn.pt')

In [ ]:
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        probs = torch.sigmoid(logits).view(-1)
        preds = (probs >= 0.5).long()

        assert preds.shape[0] == labels.shape[0]

        y_pred.extend(preds.cpu().tolist())
        y_true.extend(labels.view(-1).cpu().tolist())

print(classification_report(y_true, y_pred))
print(confusion_matrix(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.66      1.00      0.80        35
           1       0.00      0.00      0.00        18

    accuracy                           0.66        53
   macro avg       0.33      0.50      0.40        53
weighted avg       0.44      0.66      0.53        53

[[35  0]
 [18  0]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# Unfreeze last convolution block
for param in model.cnn.features[-5:].parameters():
    param.requires_grad = True

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5)
print("Last VGG block unfrozen")

Last VGG block unfrozen


In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)

In [ ]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    TP = FP = TN = FN = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.float().view(-1, 1).to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = (outputs >= 0.5).float()

        TP += ((preds == 1) & (labels == 1)).sum().item()
        TN += ((preds == 0) & (labels == 0)).sum().item()
        FP += ((preds == 1) & (labels == 0)).sum().item()
        FN += ((preds == 0) & (labels == 1)).sum().item()

    accuracy = 100 * (TP + TN) / (TP + TN + FP + FN)
    precision = TP / (TP + FP + 1e-8)
    recall = TP / (TP + FN + 1e-8)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Loss: {total_loss / len(train_loader):.4f} | "
        f"Acc: {accuracy:.2f}% | "
        f"Prec: {precision:.3f} | "
        f"Recall: {recall:.3f}"
    )
    torch.save(model,'breast-thermography_vgg_qnn_checkpoint.pt')

Epoch 1/1 | Loss: 0.6743 | Acc: 60.77% | Prec: 0.000 | Recall: 0.000


In [ ]:
torch.save(model,'breast-thermography_vgg_qnn_unfrozen.pt')

In [ ]:
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        preds = model(images)
        y_pred.extend(preds.cpu().numpy().round().astype(int).flatten())
        y_true.extend(labels.cpu().numpy().astype(int))

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))


Classification Report:

              precision    recall  f1-score   support

          -1       0.00      0.00      0.00       0.0
           0       0.00      0.00      0.00      35.0
           1       0.00      0.00      0.00      18.0

    accuracy                           0.00      53.0
   macro avg       0.00      0.00      0.00      53.0
weighted avg       0.00      0.00      0.00      53.0

Confusion Matrix:
[[ 0  0  0]
 [35  0  0]
 [18  0  0]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_